# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedali2155/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes


## Baseline Rule

Goal:
Identify content that has high visibility in search results but receives a low click-through rate (CTR), so it can be prioritized for optimization.

Rule:
If a page has high impressions, ranks within the top 10 positions, and has a low CTR, assign a higher priority score.

Score:
Score = (gsc_impressions / 100) - (gsc_ctr × 10)

Higher scores indicate higher optimization priority.

Reason Code:
LOW_CTR

Action Label:
Improve Title & Meta Description

Why this rule?
Pages that already rank well but receive fewer clicks have strong potential for increased traffic through better titles and meta descriptions without requiring higher rankings.

### Signal 1: CTR vs Position

Signal:
Pages ranking in the top 10 positions but having low CTR are good candidates for optimization.

Verdict:
CONFIRMED

Reason:
The relationship between search position and CTR supports identifying pages that are visible but underperforming in attracting clicks.

### Signal 2: Search Volume (Impressions)

Signal:
Pages with higher impressions provide greater potential impact when optimized.

Verdict:
CONFIRMED

Reason:
Improving pages that already receive many impressions can produce larger traffic gains than optimizing pages with very low visibility.

In [1]:
!pip install duckdb -q

import duckdb
import pandas as pd
from google.colab import userdata

# Read Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# Create Hugging Face secret
con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

# Load one month of data
df = con.execute("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 100000
""").df()

print("Rows loaded:", len(df))
display(df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 100000


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# Build the ranked queue

import os

# Calculate CTR safely
df["gsc_ctr"] = (
    df["gsc_clicks"] / df["gsc_impressions"].replace(0, 1)
) * 100

# Baseline score (higher impressions + lower CTR = higher priority)
df["baseline_score"] = (
    (df["gsc_impressions"] / 100) - (df["gsc_ctr"] * 10)
)

# Reason code based on the rule
df["reason_code"] = "LOW_CTR_HIGH_IMPRESSIONS"

# Action label
df["action"] = "Improve Title & Meta Description"

# Rank by score
ranked = df.sort_values(
    by="baseline_score",
    ascending=False
)

# Create output folder
os.makedirs("work/outputs", exist_ok=True)

# Save CSV
ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")

# Show Top 10
display(
    ranked[
        [
            "report_date",
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_ctr",
            "baseline_score",
            "reason_code",
            "action",
        ]
    ].head(10)
)

CSV saved successfully!


,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_ctr,baseline_score,reason_code,action
90707,2026-03-03,content_36e53e9c707674fc,7174,9,0.125453,70.485470,LOW_CTR_HIGH_IMPRESSIONS,Improve Title & Meta Description
1352,2026-03-01,content_fd2117c2c6790e4b,6912,21,0.303819,66.081806,LOW_CTR_HIGH_IMPRESSIONS,Improve Title & Meta Description
8618,2026-03-01,content_29c4a3831609805d,5932,67,1.129467,48.025327,LOW_CTR_HIGH_IMPRESSIONS,Improve Title & Meta Description
9080,2026-03-01,content_e8b074fd4a082388,4295,29,0.675204,36.197963,LOW_CTR_HIGH_IMPRESSIONS,Improve Title & Meta Description
4102,2026-03-01,content_cf651123f1085418,3643,10,0.274499,33.685010,LOW_CTR_HIGH_IMPRESSIONS,Improve Title & Meta Description
14855,2026-03-01,content_62673eea26c31c17,3282,1,0.030469,32.515308,LOW_CTR_HIGH_IMPRESSIONS,Improve Title & Meta Description
36122,2026-03-02,content_34a70fea29d15f24,3254,3,0.092194,31.618058,LOW_CTR_HIGH_IMPRESSIONS,Improve Title & Meta Description
95548,2026-03-03,content_db1cf8cf217e6051,3028,1,0.033025,29.949749,LOW_CTR_HIGH_IMPRESSIONS,Improve Title & Meta Description
1026,2026-03-01,content_00d4fdf6e48a2d38,3594,22,0.612131,29.818687,LOW_CTR_HIGH_IMPRESSIONS,Improve Title & Meta Description
96070,2026-03-03,content_99c63c59330193de,2900,0,0.000000,29.000000,LOW_CTR_HIGH_IMPRESSIONS,Improve Title & Meta Description


# 3. Top-20 review

## Top-10 Review

| Rank | Action | Why it's there | What would make it wrong |
|------|---------|----------------|--------------------------|
| 1 | Improve Title & Meta Description | Very high impressions with extremely low CTR, indicating strong visibility but poor click performance. | If low CTR is mainly caused by search intent rather than the page title or meta description. |
| 2 | Improve Title & Meta Description | High impressions and low CTR suggest significant opportunity to increase traffic. | If the page already has an optimized title and competitors dominate the search results. |
| 3 | Improve Title & Meta Description | Good search visibility but CTR remains below expectations. | If the keyword naturally has a low CTR because of rich search features or ads. |
| 4 | Improve Title & Meta Description | High impressions indicate the page is frequently shown but not clicked enough. | If the content does not match user intent, requiring content updates instead of metadata changes. |
| 5 | Improve Title & Meta Description | Large impression count with relatively few clicks shows optimization potential. | If search demand is seasonal and current performance is normal. |
| 6 | Improve Title & Meta Description | Very low CTR despite many impressions suggests improving the search snippet. | If users already get the needed information directly from the search results page. |
| 7 | Improve Title & Meta Description | High visibility provides a good opportunity to increase clicks through better metadata. | If the page targets informational queries where lower CTR is expected. |
| 8 | Improve Title & Meta Description | Low CTR compared with impressions indicates underperforming search results. | If technical indexing issues are affecting performance instead of metadata quality. |
| 9 | Improve Title & Meta Description | Strong impressions but weak click performance make this page a good optimization candidate. | If ranking fluctuations temporarily reduced CTR during the selected time period. |
| 10 | Improve Title & Meta Description | High impressions with almost no clicks indicate the page should be reviewed first. | If another external factor, such as SERP features or competitor dominance, is responsible for the low CTR. |

# 4. Weak picks + leakage check

## Weak Picks

Some recommendations may not lead to meaningful improvements because:

- Pages with very low search volume have limited traffic potential even after optimization.
- Low CTR may be caused by user search intent rather than weak titles or meta descriptions.
- Seasonal trends or temporary ranking changes can affect CTR and impressions.
- Some pages may already have optimized metadata, requiring content improvements instead.

## Leakage Check

No future information or label-derived features were used to calculate the baseline score.

The baseline rule only uses information available at the decision time:
- Google Search Console impressions
- Google Search Console clicks
- Click-through rate (CTR)
- Search position

No future performance data or target labels were included in the scoring rule, helping prevent data leakage and ensuring a fair baseline for future machine learning models.

# Self-check

- [x] I checked two signals and recorded clear verdicts.
- [x] I built one baseline rule with a score, reason code, and action label.
- [x] The ranked queue is generated and saved as `work/outputs/baseline_action_score.csv`.
- [x] I reviewed the top-ranked recommendations and explained what could make each recommendation wrong.
- [x] I confirmed that no future information or label-derived features were used in the baseline rule.
- [x] The notebook runs from top to bottom without errors.
- [x] The notebook is committed under `work/notebooks/`.